# 📒 Laboratório 2: Programação Concorrente com pthreads

## Disciplina: Programação Concorrente (ICP-361)

### Professora: Silvana Rossetto / Gabriel P. Silva
Instituto de Computação/UFRJ

## Apresentação
O objetivo deste laboratório é apresentar os conceitos de passagem de parâmetros entre *threads*

## 2. pThreads

A seguir apresentamos os conceitos básicos de programação concorrente utilizando a biblioteca `pThreads`. Funções principais utilizadas são:

  1. Cria uma nova thread.

``` c
int pthread_create(pthread_t *thread, const pthread_attr_t *attr,  void *(*start_routine) (void *), void *arg);
```

**Uso e Explicação:**

Esta função é utilizada para criar uma nova *thread* dentro do processo atual. A nova *thread* começa sua execução invocando a função passada como argumento.

  - thread: Um ponteiro para uma variável do tipo `pthread_t`. Se a criação for bem-sucedida, a função armazenará o identificador (ID) da nova *thread* neste endereço.

  - attr: Um ponteiro para uma estrutura `pthread_attr_t` que define os atributos da thread (como tamanho da pilha, política de escalonamento, etc.). Na maioria dos casos simples, passa-se **NULL** para usar os atributos padrão.

  - start_routine: Um ponteiro para a função que a *thread* irá executar. Essa função obrigatoriamente deve retornar um `void*` e receber um único parâmetro `void*`.

  - arg: O argumento que será passado para a função `start_routine`. Como é um `void*`, você pode passar o endereço de qualquer tipo de dado (um inteiro, uma *struct*, etc.) e fazer o cast correto dentro da função. Se a função não precisar de argumentos, passe **NULL**.
  
  - Retorno: Retorna 0 em caso de sucesso. Se houver erro (por exemplo, falta de recursos do sistema), retorna um código de erro numérico.

  2. Encerra a thread atual.

```c
void pthread_exit(void *retval);
```
**Uso e Explicação:**

Esta função encerra a execução da *thread* que a chamou. É uma forma de finalizar a *thread* de maneira limpa, sem encerrar o processo inteiro (como aconteceria se você chamasse exit() da biblioteca padrão do C).

  - retval: Um ponteiro (`void*`) para o valor de retorno da *thread*. Esse valor fica guardado pelo sistema operacional e pode ser recuperado por outra *thread* que chame `pthread_join` esperando por esta. Se você não precisa retornar nada, pode passar **NULL**.
  
  - Retorno: Esta função não retorna para o chamador, pois a *thread* deixa de existir imediatamente após a chamada.

Nota importante: Se a *thread* principal (a que roda a função main) chamar `pthread_exit`, ela é encerrada, mas o processo em si continua vivo até que todas as outras *threads* criadas também terminem.

  3. Aguarda o término de uma thread específica.

```c
int pthread_join(pthread_t thread, void **retval);
```

**Uso e Explicação:**

Esta função bloqueia a execução da *thread* que a chamou até que a *thread* alvo especificada termine sua execução (ou seja, até que a *thread* alvo chame `pthread_exit` ou retorne de sua função principal). É o equivalente ao `wait()` usado para processos.

  - thread: O identificador (ID) da *thread* que você deseja aguardar (o mesmo ID que foi preenchido por pthread_create).

  - retval: Um ponteiro para um ponteiro (`void**`). Se não for **NULL**, a função `pthread_join` copiará o valor de retorno da *thread* alvo (aquele passado no `pthread_exit`) para o local apontado por `retval`. Isso permite recuperar resultados processados pela `thread`. Se você não se importa com o valor de retorno, pode passar NULL.
  
  - Retorno: Retorna `0` em caso de sucesso e um código de erro caso falhe (por exemplo, se tentar fazer join em uma *thread* que não existe ou que não é *joinable*).

Nota importante: Fazer o `join` também instrui o sistema operacional a liberar os recursos (como a pilha) que a *thread* estava usando. Se você não fizer o `join` em uma *thread* (e não a criar como \textit{detached}), ocorrerá um vazamento de recursos conhecido como *zombie thread*.

## 3. Atividades Iniciais (Revisão de Exemplos)
### Atividade 1: Criação de threads com `hello.c`

#### **Objetivo**: Mostrar a criação básica de *threads*.

In [267]:
%%writefile hello.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Gabriel P. Silva */
/* Laboratório: 2 */
/* Codigo: "Hello World" usando threads em C */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

//--funcao executada pelas threads
void *PrintHello (void *arg) {
   printf("Alô galera do Fundão! \n");
   pthread_exit(NULL);
}

//--funcao principal do programa
int main(int argc, char* argv[]) {
   int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

   //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
   if(argc<2) {
      printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
      return 1;
   }
   nthreads = atoi(argv[1]);

   //identificadores das threads no sistema
   pthread_t tid[nthreads];

   //cria as threads e atribui a tarefa de cada thread
   for(long int i=0; i < nthreads; i++) {
      printf("--Cria a thread %ld\n", i);
      if (pthread_create(&tid[i], NULL, PrintHello, &i)) {
         printf("--ERRO: pthread_create()\n");
         return 2;
      }
   }

   //log da funcao main
   printf("--Thread principal terminou\n");

   for(int i=0; i<nthreads; i++){
      pthread_join(tid[i], NULL);
   }

   pthread_exit(NULL); //necessario para que o processo não seja encerrado antes das threads terminarem
                       /* Não é uma forma canonica -- deve-se usar pthread_join */
}

Overwriting hello.c


In [268]:
# Célula de Código: Exemplo de compilação e execução
!gcc -o hello hello.c -Wall -lpthread
!./hello 4

--Cria a thread 0
--Cria a thread 1
Alô galera do Fundão! 
--Cria a thread 2
Alô galera do Fundão! 
Alô galera do Fundão! 
--Cria a thread 3
--Thread principal terminou
Alô galera do Fundão! 


Questão: O resultado impresso muda de uma execução para outra?

### Atividade 2: Passagem de argumentos (helloArg.c)

#### Objetivo: Mostrar como passar *um* argumento simples para uma thread

In [269]:
%%writefile helloArg.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C com passagem de um argumento */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

#define TAMANHOS

//--funcao executada pelas threads
void *PrintHello (void* arg) {
   //typecasting do argumento
   int idThread = *(int*) arg;
   //log da thread
   printf("Hello World da thread: %d\n", idThread);

   pthread_exit(NULL);
}

//--funcao principal do programa
int main(int argc, char * argv[]) {
   int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

   //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
   if(argc<2) {
       printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
       return 1;
   }
   nthreads = atoi(argv[1]);

   //identificadores das threads no sistema
   pthread_t tid[nthreads];

#ifdef TAMANHOS
   //exibe os tamanhos dos tipos - para informacao apenas
   puts("Tamanhos dos tipos de dados:");
   printf("Tamanho ponteiro:%ld bytes\n", sizeof(void*));
   printf("Tamanho int:%ld bytes\n", sizeof(int));
   printf("Tamanho long int:%ld bytes\n\n", sizeof(long int));
#endif

   //cria as threads e atribui a tarefa de cada thread
   for(long int i=0; i<nthreads; i++) {
      printf("--Cria a thread %ld\n", i);
      if (pthread_create(&tid[i], NULL, PrintHello, (void*) &i)) {
         printf("--ERRO: pthread_create()\n");
	       return 2;
      }
   }

   //log da função main
   printf("--Thread principal terminou\n");
   pthread_exit(NULL); //necessario para que o processo não seja encerrado antes das threads terminarem
                       // Não é uma forma canonica também --> deve-se usar pthread_join()
   //return 0;
}

Overwriting helloArg.c


In [270]:
!gcc -o teste helloArg.c -Wall -lpthread
!./teste 4

Tamanhos dos tipos de dados:
Tamanho ponteiro:8 bytes
Tamanho int:4 bytes
Tamanho long int:8 bytes

--Cria a thread 0
--Cria a thread 1
--Cria a thread 2
--Cria a thread 3
--Thread principal terminou
Hello World da thread: 4
Hello World da thread: 4
Hello World da thread: 4
Hello World da thread: 4


### Questão: Por que foi necessário usar `long int` na variável iteradora?

R: Esse é um método não canônico de passagem de parâmetro por valor para a thread. Diversas adaptações são necessárias no código, tais como declarar o índice do laço como long int, para que tenha o mesmo tamanho do ponteiro, que é de 64 bits.

### Atividade 3: Modificar o programa helloArg.c para que a passagem de parâmetros seja feita corretamente, ou seja, por referência. Verificar se o funcioamento programa está correto.

### Atividade 4: Múltiplos argumentos (helloArgs.c)

#### Objetivo:
Mostrar o uso de estruturas para múltiplos argumentos.

In [271]:
%%writefile helloArgs.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C passando mais de um argumento */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

//cria a estrutura de dados para armazenar os argumentos da thread
typedef struct {
   int idThread, nThreads;
} t_Args;

//--funcao executada pelas threads
void *PrintHello (void *arg) {
  //typecarting do argumento
  t_Args args = *(t_Args*) arg;
  t_Args *args2 = (t_Args*) arg;

  //log da thread
  printf("Sou a thread %d de %d threads\n", args.idThread, args.nThreads);
  printf("Sou a thread %d de %d threads\n", args2->idThread, args2->nThreads);

  free(arg); //libera a memoria que foi alocada na main

  pthread_exit(NULL);
}

//--funcao principal do programa
int main(int argc, char* argv[]) {
  t_Args *args; //receberá os argumentos para a thread
  int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

  //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
  if(argc<2) {
      printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
      return 1;
  }
  nthreads = atoi(argv[1]);

  //identificadores das threads no sistema
  pthread_t tid_sistema[nthreads];

  //cria as threads
  for(int i=1; i<=nthreads; i++) {
    printf("--Aloca e preenche argumentos para thread %d\n", i);
    args = malloc(sizeof(t_Args));
    if (args == NULL) {
      printf("--ERRO: malloc()\n");
      return 2;
    }
    args->idThread = i;
    args->nThreads = nthreads;

    printf("--Cria a thread %d\n", i);
    if (pthread_create(&tid_sistema[i-1], NULL, PrintHello, (void*) args)) {
      printf("--ERRO: pthread_create() da thread %d\n", i);
    }
  }

  //log da função principal
  printf("--Thread principal terminou\n");

  pthread_exit(NULL);  /* Não é uma forma canonica de terminar o programa principal */
}

Overwriting helloArgs.c


In [272]:
!gcc -o teste helloArgs.c -Wall -lpthread
!./teste 3

--Aloca e preenche argumentos para thread 1
--Cria a thread 1
--Aloca e preenche argumentos para thread 2
--Cria a thread 2
--Aloca e preenche argumentos para thread 3
--Cria a thread 3
Sou a thread 2 de 3 threads
Sou a thread 2 de 3 threads
Sou a thread 1 de 3 threads
Sou a thread 1 de 3 threads
--Thread principal terminou
Sou a thread 3 de 3 threads
Sou a thread 3 de 3 threads


### Questão: Por que foi necessário criar uma estrutura de dados nova?

\### Atividade 4: Sincronização com pthread_join (helloJoin.c)

#### Objetivo: Fazer a thread principal (main) aguardar as threads criadas

In [273]:
%%writefile helloJoin.case
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C e a funcao que espera as threads terminarem */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

//cria a estrutura de dados para armazenar os argumentos da thread
typedef struct {
   int idThread, nThreads;
} t_Args;

//funcao executada pelas threads
void *PrintHello (void *arg) {
  t_Args *args = (t_Args *) arg;

  printf("Sou a thread %d de %d threads\n", args->idThread, args->nThreads);
  free(arg); //libera a alocacao feita na main

  pthread_exit(NULL);
}

//funcao principal do programa
int main(int argc, char* argv[]) {
  t_Args *args; //receberá os argumentos para a thread

  int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

  //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
  if(argc<2) {
      printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
      return 1;
  }
  nthreads = atoi(argv[1]);

  //identificadores das threads no sistema
  pthread_t tid_sistema[nthreads];

  //cria as threads
  for(int i=1; i<=nthreads; i++) {
    printf("--Aloca e preenche argumentos para thread %d\n", i);
    args = malloc(sizeof(t_Args));
    if (args == NULL) {
      printf("--ERRO: malloc()\n");
      return 1;
    }
    args->idThread = i;
    args->nThreads = nthreads;

    printf("--Cria a thread %d\n", i);
    if (pthread_create(&tid_sistema[i-1], NULL, PrintHello, (void*) args)) {
      printf("--ERRO: pthread_create()\n");
      return 2;
    }
  }

  //espera todas as threads terminarem
  for (int i=0; i<nthreads; i++) {
    if (pthread_join(tid_sistema[i], NULL)) {
         printf("--ERRO: pthread_join() da thread %d\n", i);
    }
  }

  //log da função principal
  printf("--Thread principal terminou\n");

  //pthread_exit(NULL); //nao é necessario nesse caso, por que?
}

Overwriting helloJoin.case


### Atividade 5: Retorno de parâmetro

#### Modificar o programa anterior, de modo que a estrutura definida tenha parâmetros de retorno também.

### Atividade 6: Implementação Prática

#### Desafio:
  - Implementar um programa concorrente com $M$ threads para somar 1 a cada elemento de um vetor de $N$ elementos.
  - Calcular a soma de todos os elementos do vetor, antes e depois da adição, e retornar os valores parciais para o programa principal.
  - Receber os parâmetros no programa principal, totalizar e verificar se houve algum erro de execução.

#### Requisitos:
  - $M$ e $N$ passados como argumentos de linha de comando.
  - Divisão balanceada de carda.
  - O programa deve ter duas versês: na primeira divisão do vetor deve ser feita em bloco e na segunda deve ser feita alternadamente, como apresentado em sala de aula.
  - Implementar funções de inicialização e verificação de resultados.
  - Executar o programa com o maior número possível de threads e verificar o ganho final obtido. Utilize as rotinas.

In [274]:
%%writefile exercicio5.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <time.h> // Para medir o ganho

typedef struct{
  long int idThread;
  long int tam_vet;
  long int inicio;
  long int fim;

} t_Args;

typedef struct{
  long int somaAntes;
  long int somaDepois;
} t_Retorno;

int *vetor;

void* tarefa(void* arg){
  t_Args* args = (t_Args*) arg;
  t_Retorno* retorno = malloc(sizeof(t_Retorno));
  retorno->somaAntes = 0;
  retorno->somaDepois = 0;
  for(int i = args->inicio; i < args->fim; i++){
    retorno->somaAntes += vetor[i];
    vetor[i]++;
    retorno->somaDepois += vetor[i];
  }
  free(arg);
  //printf("thread %ld -- soma %ld\n", args->idThread, retorno->somaDepois);
  pthread_exit((void *) retorno);
}

int main(int argc, char* argv[]){
  int nThreads, tamVet, somaAntes = 0, somaDepois = 0;
  t_Retorno *retorno;

  if(argc<3){
    printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
    return 1;
  }
  nThreads = atoi(argv[1]);
  tamVet = atoi(argv[2]);

  vetor = malloc(tamVet * sizeof(int));

  for(int i = 0; i < tamVet; i++){
    vetor[i] = 0;
  }

  pthread_t tid[nThreads];

  struct timespec start, end;
  clock_gettime(CLOCK_MONOTONIC, &start);

  for(int i = 0; i < nThreads; i++){
    t_Args *args = malloc(sizeof(t_Args));
    args->idThread = i;
    args->tam_vet = tamVet;
    args->inicio = i * (tamVet / nThreads);
    args->fim = (i < nThreads-1)? (i + 1) * (tamVet / nThreads) : tamVet;
    if(pthread_create(&tid[i], NULL, tarefa, (void*) args)){
      printf("--ERRO: pthread_create()\n");
      return 2;
    }
  }

  for(int i = 0; i < nThreads; i++){
    if(pthread_join(tid[i], (void *) &retorno)){
      printf("--ERRO: pthread_join()\n");
      return 3;
    }
    somaAntes += retorno->somaAntes;
    somaDepois += retorno->somaDepois;
    free(retorno);
  }

  clock_gettime(CLOCK_MONOTONIC, &end);
  double elapsed = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;

  if (somaDepois == (somaAntes + tamVet)){
    printf("%d == %d\nO vetor está correto\nTempo %.6f segundos\n", (somaAntes + somaDepois), tamVet, elapsed);
  }
  else{
    printf("%d == %d\nO vetor não está correto\n", (somaAntes + somaDepois), tamVet);
  }

  free(vetor);
}

Overwriting exercicio5.c


In [275]:
# Compile o seu programa
!gcc -o exercicio5 exercicio5.c -Wall -lpthread

# Execute com M=4 threads e N=1000 elementos
!./exercicio5 12 1000000

1000000 == 1000000
O vetor está correto
Tempo 0.005063 segundos


In [276]:
%%writefile exercicio5.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <time.h> // Para medir o ganho

typedef struct{
  long int idThread;
  long int tam_vet;
  long int nThreads;
} t_Args;

typedef struct{
  long int somaAntes;
  long int somaDepois;
} t_Retorno;

int *vetor;

void* tarefa(void* arg){
  t_Args* args = (t_Args*) arg;
  t_Retorno* retorno = malloc(sizeof(t_Retorno));
  retorno->somaAntes = 0;
  retorno->somaDepois = 0;
  for(long int i = args->idThread; i < args->tam_vet; i = i + (args->nThreads)){
    retorno->somaAntes += vetor[i];
    vetor[i]++;
    retorno->somaDepois += vetor[i];
  }
  free(arg);
  pthread_exit((void *) retorno);
}

int main(int argc, char* argv[]){
  int nThreads, tamVet, somaAntes = 0, somaDepois = 0;
  t_Retorno *retorno;

  if(argc<3){
    printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
    return 1;
  }
  nThreads = atoi(argv[1]);
  tamVet = atoi(argv[2]);

  vetor = malloc(tamVet * sizeof(int));

  for(int i = 0; i < tamVet; i++){
    vetor[i] = 0;
  }

  pthread_t tid[nThreads];

  struct timespec start, end;
  clock_gettime(CLOCK_MONOTONIC, &start);

  for(long int i = 0; i < nThreads; i++){
    t_Args *args = malloc(sizeof(t_Args));
    args->idThread = i;
    args->tam_vet = tamVet;
    args->nThreads = nThreads;
    if(pthread_create(&tid[i], NULL, tarefa, (void*) args)){
      printf("--ERRO: pthread_create()\n");
      return 2;
    }
  }

  for(int i = 0; i < nThreads; i++){
    if(pthread_join(tid[i], (void *) &retorno)){
      printf("--ERRO: pthread_join()\n");
      return 3;
    }
    somaAntes += retorno->somaAntes;
    somaDepois += retorno->somaDepois;
    free(retorno);
  }

  clock_gettime(CLOCK_MONOTONIC, &end);
  double elapsed = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;

  if (somaDepois == (somaAntes + tamVet)){
    printf("%d == %d\nO vetor está correto\nTempo %.6f segundos\n", (somaAntes + somaDepois), tamVet, elapsed);
  }
  else{
    printf("%d == %d\nO vetor não está correto\n", (somaAntes + somaDepois), tamVet);
  }

  free(vetor);
}

Overwriting exercicio5.c


In [277]:
# Compile o seu programa
!gcc -o exercicio5 exercicio5.c -Wall -lpthread

# Execute com M=4 threads e N=1000 elementos
!./exercicio5 4 1000000

1000000 == 1000000
O vetor está correto
Tempo 0.005265 segundos
